# SASV: ECAPA + WavLM score-sum (**dev**)

Same fusion as notebook 03, with CM hardcoded to your app **WavLM** LA detector:

```text
s_sasv = s_asv + (1 - P_spoof)
```

- Use lab LA flacs (browser ~1.0 issue does not apply here).
- Compare against `runs/ecapa_only_dev/` and `runs/ecapa_plus_lfcc_dev/`.
- Tune / decide on **dev** here; locked **eval** is notebook `06`.

Needs `transformers` in the `app/server` kernel.

In [5]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "score_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from score_lib import score_fused_trials

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA exists:", DEFAULT_LA.exists())
print("SASV exists:", DEFAULT_SASV.exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA exists: True
SASV exists: True


## Knobs

- `SMOKE = True` → 500 stratified trials (sanity check)
- `SMOKE = False` → full **dev** (~29k)
- `CM_BACKEND` is fixed to **`wavlm`**

In [6]:
SMOKE = False
SPLIT = "dev"
MAX_TRIALS = 500 if SMOKE else 0
CM_BACKEND = "wavlm"
DEVICE = "cuda"
FORCE_CPU = False

assert CM_BACKEND == "wavlm"
assert SPLIT == "dev"

## Run ECAPA + WavLM fusion (dev)

Writes `runs/ecapa_plus_wavlm_dev/`

In [7]:
summary = score_fused_trials(
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    split=SPLIT,
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    cm_backend=CM_BACKEND,
    output_dir=RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_{SPLIT}",
)
{
    "system": summary["system"],
    "sasv_eer_%": summary["sasv_eer_percent"],
    "sv_eer_%": summary["sv_eer_percent"],
    "spf_eer_%": summary["spf_eer_percent"],
    "n": summary["num_scored"],
}

Trials: {'target': 1484, 'nontarget': 5768, 'spoof': 22296, 'total': 29548} | CM=wavlm


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Enrol dev:   0%|          | 0/10 [00:00<?, ?it/s]

Score fused:   0%|          | 0/29548 [00:00<?, ?it/s]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


{
  "system": "ecapa_plus_wavlm_sum",
  "split": "dev",
  "max_trials": 0,
  "num_scored": 29548,
  "key_counts": {
    "target": 1484,
    "nontarget": 5768,
    "spoof": 22296,
    "total": 29548
  },
  "device": "cuda",
  "cm_backend": "wavlm",
  "fusion": "s_asv + (1 - p_spoof)",
  "sasv_eer": 0.07345013477088944,
  "sv_eer": 0.118598382749326,
  "spf_eer": 0.03840970350404319,
  "sasv_eer_percent": 7.345013477088943,
  "sv_eer_percent": 11.859838274932601,
  "spf_eer_percent": 3.8409703504043193
}


{'system': 'ecapa_plus_wavlm_sum',
 'sasv_eer_%': 7.345013477088943,
 'sv_eer_%': 11.859838274932601,
 'spf_eer_%': 3.8409703504043193,
 'n': 29548}

## Compare with ECAPA-only / LFCC (dev)

In [8]:
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_dev" / "metrics_dev.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / "ecapa_plus_lfcc_dev" / "metrics_dev.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / "ecapa_plus_wavlm_dev" / "metrics_dev.json"),
]:
    if not path.exists():
        print("Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"{label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%"
    )

ecapa_only            SASV=15.2291%  SV=1.2483%  SPF=17.9090%
ecapa_plus_lfcc       SASV=1.1438%  SV=2.0978%  SPF=0.0897%
ecapa_plus_wavlm      SASV=7.3450%  SV=11.8598%  SPF=3.8410%


## Next

1. Set `SMOKE = False` and re-run for full **dev**.
2. If you keep WavLM for reporting, run `06_ecapa_plus_wavlm_eval.ipynb` **once** (no further tuning).